In [1]:
import shutil
import pandas as pd
import random

import copy
import numpy as np # Import numpy for checking finite values
import matplotlib.pyplot as plt # Import matplotlib for potential debugging

import os
import math # Import math for ceil
from sklearn.manifold import TSNE # Import TSNE to check default perplexity



import os
import glob
import json
import numpy as np
import tensorflow as tf




In [146]:
import sys
utils_path = 'C:\\Users\\admin\\0.Master_Thesis\\'
sys.path.append(utils_path)


import sys
import os
import importlib
# Aggiungi la directory principale del progetto al percorso di sistema
path = 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn'
path_unfixed = 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files'
if path not in sys.path:
    sys.path.append(path)
# Ricaricali per forzare la lettura aggiornata del file


# Importa tutte le funzioni e gli oggetti dal modulo trainer
# Attenzione: l'uso di '*' non è sempre raccomandato in codice di produzione
# perché rende meno chiaro da dove provengano le funzioni.
from utils import trainer, DensityDifference
#import trainer, DensityDifference
importlib.reload(trainer)
importlib.reload(DensityDifference)

from utils import trainer, DensityDifference, dataset


In [147]:
decache_files = ['utils', 'cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 'Dataset_Elaboration_Class', 'Dataset_utils', 'cellCnn.downsample_new_unfixed']
# Rimuovi il modulo specifico dalla cache

from Dataset_utils import remove_from_cache
remove_from_cache(decache_files)

from utils import trainer, DensityDifference

importlib.reload(trainer)
importlib.reload(DensityDifference)
importlib.reload(dataset)

from utils import trainer, DensityDifference, dataset
#import train_logistic_cv


from Dataset_Elaboration_Class import class_division, donor_division, splitting, patient_code_extraction
from Dataset_Elaboration_Class import run_CellCNN_res, dataset_elaboration, CellCNN_restructured, CV_CellCNN_restructured, CV_best_acc

from Dataset_utils import extract_hyper, phenotype_prediction, default_serializer, remove_from_cache, show_hyperparameters, min_length
from Dataset_utils import elaborate_predictions, show_hyper

utils rimosso dalla cache
cellCnn.model_new_unfixed non trovato nella cache
cellCnn.utils_new_unfixed non trovato nella cache
Dataset_Elaboration_Class non trovato nella cache
Dataset_utils rimosso dalla cache
cellCnn.downsample_new_unfixed non trovato nella cache


In [4]:
%%time

# Trova tutti i file con estensione specifica in una cartella
data_folder = f'{utils_path}B-ALL_Datasets'
extension = '*.csv'  # cambia con l'estensione che ti serve

files_list = glob.glob(os.path.join(data_folder, extension))

ALL_DATASETS = []
counter = 0
multiple_donations = {}
no_id = []
for file_path in files_list:
    #import the dataset
    data = pd.read_csv(file_path, sep = ';', decimal = ',').astype('float32')
    ALL_DATASETS.append(data) # list of all datasets

    # divide the datasets by donors
    multiple_donations = patient_code_extraction(file_path, counter, multiple_donations)
    
    print(f"Elaborando file {counter}: {file_path}") # information about the process
    
    counter += 1 

# Fix no_id datasets
last_identifier = 0
for element in multiple_donations.keys():
    if element.isdigit():
        if int(element) > int(last_identifier):
            last_identifier = int(element)
print(last_identifier)
for data in multiple_donations['no_id']:
    last_identifier += 1
    multiple_donations[str(last_identifier)] = [data]
multiple_donations.pop('no_id')

Elaborando file 0: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220204-2900.csv
Elaborando file 1: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220208-3665.csv
Elaborando file 2: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220216-3546.csv
Elaborando file 3: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D15_1.csv
Elaborando file 4: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D29_1.csv
Elaborando file 5: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D71_1.csv
Elaborando file 6: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D15_2.csv
Elaborando file 7: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D29_1.csv
Elaborando file 8: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE1_D15_2.csv
Elaborando file 9: C

[0, 1, 2]

In [5]:
for i, data in enumerate(ALL_DATASETS):
    blast_n = (dataset['IsBlast'] == 1).sum()
    print(f'data {i} has {blast_n} cells over {len(dataset)} cells')

dataset 0 has 0 cells over 6558750 cells
dataset 1 has 0 cells over 2764877 cells
dataset 2 has 0 cells over 1291729 cells
dataset 3 has 1348 cells over 843500 cells
dataset 4 has 170 cells over 1404000 cells
dataset 5 has 0 cells over 3265250 cells
dataset 6 has 639 cells over 430869 cells
dataset 7 has 15 cells over 722000 cells
dataset 8 has 757 cells over 864000 cells
dataset 9 has 0 cells over 1947518 cells
dataset 10 has 308059 cells over 778750 cells
dataset 11 has 830101 cells over 5510750 cells
dataset 12 has 72 cells over 208000 cells
dataset 13 has 0 cells over 2912500 cells
dataset 14 has 3096 cells over 160500 cells
dataset 15 has 1449 cells over 3591480 cells
dataset 16 has 0 cells over 3107000 cells
dataset 17 has 9147 cells over 637409 cells
dataset 18 has 227 cells over 2928000 cells
dataset 19 has 518 cells over 164553 cells
dataset 20 has 398 cells over 191975 cells
dataset 21 has 0 cells over 169000 cells
dataset 22 has 2390 cells over 479000 cells
dataset 23 has 77

In [6]:
# Show patients donations
print(multiple_donations)
#['12', '9', '13', '7', '11'] 

{'11': [3, 4, 5], '12': [6, 7], '1': [8, 9], '2': [10, 11], '3': [12, 13], '4': [14, 15, 16], '6': [17, 18], '7': [19, 20, 21], '8': [22, 23], '9': [24, 25], '13': [0], '14': [1], '15': [2]}


In [7]:
healthy_donors, blast_donors, mixed_donors = donor_division(multiple_donations, ALL_DATASETS)
#print(first_subset)
#testing_set_idx = list(set(range(len(files_list))) - set(training_set_idx) - set(val_set_idx))
#print(training_set_idx, val_set_idx, testing_set_idx)
print(healthy_donors, blast_donors, mixed_donors)

{'11': [1, 1, 0], '12': [1, 1], '1': [1, 0], '2': [1, 1], '3': [1, 0], '4': [1, 1, 0], '6': [1, 1], '7': [1, 1, 0], '8': [1, 1], '9': [1, 1], '13': [0], '14': [0], '15': [0]}
['12', '2', '6', '8', '9'] ['13', '14', '15'] ['11', '1', '3', '4', '7']


In [93]:
def dataset_elaboration(multiple_donations, ALL_DATASETS, healthy_donors, blast_donors,
                        mixed_donors, n_sub = 3, seed = 42):
    """ Samples donors for Train, Validation and Test sets"""
    
    train_donors = []
    val_donors = []
    test_donors = []
    
    random.seed(seed)
    print(f'Precess starts. Dividing donors...')
    
    # sammple indexed for donor division
    healthy_donors_idx = random.sample(list(range(len(healthy_donors))), len(healthy_donors))
    blast_donors_idx = random.sample(list(range(len(blast_donors))), len(blast_donors))
    mixed_donors_idx = random.sample(list(range(len(mixed_donors))), len(mixed_donors))
    print(f'healthy_donors_idx, blast_donors_idx, mixed_donors_idx: {healthy_donors_idx}, {blast_donors_idx},{mixed_donors_idx}')

    print(f'Seting Train, Validation and Test idx...')
    # just divide accoding to the sampled indexes
    train_donors_idx, val_donors_idx, test_donors_idx = splitting(healthy_donors, blast_donors, mixed_donors, healthy_donors_idx, blast_donors_idx, mixed_donors_idx)
    print(train_donors_idx, val_donors_idx, test_donors_idx)

    return train_donors_idx, val_donors_idx, test_donors_idx

def donation_extraction(donors_idx, multiple_donations, ALL_DATASETS):
    """ Retrieves specific donors datasets (for ex. donors for train) from all datasets list """
    datasets_extracted = []
    for donor in donors_idx:
        donor_datasets = multiple_donations[donor]
        #print(donor_datasets)
        donor_donations = []
        for donation in donor_datasets:
            donation_dataset = ALL_DATASETS[donation].drop(columns = ['Original_ID'])
            
            donor_donations.append(donation_dataset)
            
        datasets_extracted.append(donor_donations)
    return datasets_extracted

def B_H_data_extraction(dataset, blast = True):
    """ Extracts blast or healthy cells from a specific dataset """
    code = 1 if blast == True else 0 
    data = pd.DataFrame()

    donation = dataset
    sub_data = donation[donation['IsBlast'] == code]
 
    data = pd.concat([data, sub_data], ignore_index = True)

    return data

def data_for_chunk(dataset, n_sub, blast = True):
    """ Compute the length of a chunk for Blast or Healthy datasets"""
    chunk_length = int(len(dataset) / n_sub)
              
    return chunk_length

def chunks_division(dataset, n_sub, blast = True, seed = 42):
    """ divides the dataset  in chunks creating a list of subsets of the dataset """
    if not blast:
        n_sub = n_sub*2

    
    chunk_length = data_for_chunk(dataset, n_sub = n_sub, blast = blast)
    
    #randomize data order
    dataset_shuffled = dataset.sample(frac=1, random_state=seed).reset_index(drop=True) # setting drop = False, it creates a new column with the original index of the ell

    data_chunks = [] 
    for i in range(n_sub):
        chunk_i = dataset_shuffled.iloc[i*chunk_length:  chunk_length * (i+1)]
        data_chunks.append(chunk_i)

    return data_chunks

def mixed_build_datasets(healthy_data_chunks, blast_data_chunks, n_cells = 100000, seed = 42):
    """ Builds the final datasets of a specific donor
        n_cells = number of healthy cells"""
    new_donor_datasets = []
    new_donor_y = []
    # blast_percentage_choice
    blast_percentages = [0.005, 0.01, 0.05, 0.1] #, 0.2] # 
    

    # dataset with blast cells
    for i in range(n_sub):
        # set randomicity
        seed = seed * (i+1) + 10
        random.seed(seed)
        n_healthy_cells = len(healthy_data_chunks[i]) # number of healthy cells
        
        blast_perc = random.choice(blast_percentages)
        n_blast_cells = int(blast_perc * n_cells) # number of blast cells
        
        # if number of cells is too high, the sample with replacement is activated
        if n_blast_cells > len(blast_data_chunks[i]):
            b_rep = True
        else:
            b_rep = False
    
        blast_data = blast_data_chunks[i].sample(n = n_blast_cells, replace = b_rep, random_state = seed).reset_index(drop=True)

        
        if n_cells > len(healthy_data_chunks[i]):
            h_rep = True
        else:
            h_rep = False
        healthy_data = healthy_data_chunks[i].sample(n = n_cells, replace = h_rep, random_state = seed).reset_index(drop=True)
        new_blast_dataset_i = pd.concat([healthy_data, blast_data], ignore_index = True)
        new_donor_datasets.append(new_blast_dataset_i)
        new_donor_y.append(1)
        print(f'New Artificial Blast Donation {i + 1}: length = {len(new_blast_dataset_i)}, healthy cells:{n_healthy_cells}, blast cells: {len(blast_data)}')

    # dataset with only healthy cells
    for i in range(n_sub, int(n_sub*2)):
        n_healthy_cells = len(healthy_data_chunks[i]) # number of healthy cells
        
        if n_cells > len(healthy_data_chunks[i]):
            h_rep = True
        else:
            h_rep = False
        # set randomicity
        seed = seed * (i+1) + 10
        random.seed(seed)
        
        
        healthy_data_i = healthy_data_chunks[i].sample(n = n_cells, replace = h_rep, random_state = seed).reset_index(drop=True)
        new_donor_datasets.append(healthy_data_i)
        new_donor_y.append(0)
        print(f'New Artificial Healthy Donation {i - n_sub + 1}: length = {len(healthy_data_i)}')

    return new_donor_datasets, new_donor_y
def healthy_build_datasets(healthy_data_chunks, n_cells = 100000, seed = 42):
    """ Builds the final datasets of a specific donor """
    new_donor_datasets = []
    new_donor_y = []
    # dataset with only healthy cells
    for i in range(n_sub, n_sub*2):
        # set randomicity
        seed = seed * (i+1) + 10
        random.seed(seed)
        
        n_healthy_cells = len(healthy_data_chunks[i]) # number of healthy cells
        
        if n_cells > len(healthy_data_chunks[i]):
            h_rep = True
        else:
            h_rep = False
        
        healthy_data_i = healthy_data_chunks[i].sample(n = n_cells, replace = h_rep, random_state = seed).reset_index(drop=True)
        new_donor_datasets.append(healthy_data_i)
        new_donor_y.append(0)
        print(f'New Artificial Healthy Donation {i - n_sub + 1}: length = {len(healthy_data_i)}')

    return new_donor_datasets, new_donor_y

def check_dataset_types(donor_datasets):
    counter = 0
    for data in donor_datasets:
        if len(data[data['IsBlast'] == 1]) > 0:
            counter += 1
    
    if counter > 0:
        type = 1
    else:
        type = 0
        
    #print(type)
    #print(f'\n')
    return type


def generate_new_datasets(donor_datasets_extracted, n_sub, n_cells, seed):
    """ generates new datasets from multiple donations of the same donor """
    
    blast_data = pd.DataFrame()
    healthy_data = pd.DataFrame()
   
    type = check_dataset_types(donor_datasets_extracted)
    #print(type)
    # aggregate healthy and blast data form all donor cells
    for dt in donor_datasets_extracted:
        healthy = len(dt[dt['IsBlast'] == 0])
        #print(healthy)
        if type == 1:
            blast_dataset_i = B_H_data_extraction(dt) #blast_data
            blast_data = pd.concat([blast_data, blast_dataset_i], ignore_index = True)

        
        healthy_dataset_i = B_H_data_extraction(dt, False)  #healthy_data
        #print(len(healthy_dataset_i))
        # create a single big dataset of blast or healthy cells
        healthy_data = pd.concat([healthy_data, healthy_dataset_i], ignore_index = True)
        #print(len(healthy_data))
    
    # shuffle anf divide the two datasets in chunks, from which extract final data
    if type == 1:
        blast_data_chunks = chunks_division(blast_data, n_sub = n_sub, blast = True, seed = seed)
        healthy_data_chunks = chunks_division(healthy_data, n_sub = n_sub, blast = False, seed = seed)
        
        #print(f'vhuk:{len(healthy_data_chunks[0])}')
    
        # create the new datasets
        new_donor_datasets, new_donor_y = mixed_build_datasets(healthy_data_chunks, blast_data_chunks, n_cells, seed = seed)
        
        
    else:
        healthy_data_chunks = chunks_division(healthy_data, n_sub = n_sub, blast = False, seed = seed)
        #print(len(healthy_data_chunks[0]))
        new_donor_datasets, new_donor_y = healthy_build_datasets(healthy_data_chunks, n_cells, seed = seed)

    return new_donor_datasets, new_donor_y



def splitting_and_dataset_elaboration(train_datasets_extracted, val_datasets_extracted, test_datasets_extracted, n_sub, n_cells, seed):
    new_train_datasets = []
    new_train_y = []
    
    new_val_datasets = []
    new_val_y = []
    
    new_test_datasets = []
    new_test_y = []

    print(f'New training datasets creation...')
    for donor_datasets in train_datasets_extracted:
        print(f'\nNew Donor')
        
        gen_results = generate_new_datasets(donor_datasets, n_sub, n_cells, seed)
        #for dt in gen_results
        new_train_datasets += gen_results[0]
        new_train_y += gen_results[1]
        seed += 1 
    print(new_train_y )
    print(f'Done!\n')
    
    
    print(f'New training datasets creation...')
    for donor_datasets in val_datasets_extracted:
        print(f'\nNew Donor')
        
        gen_results = generate_new_datasets(donor_datasets, n_sub, n_cells, seed)
        new_val_datasets += gen_results[0]
        new_val_y += gen_results[1]
        seed += 1 
    print(new_val_y )
    print(f'Done!\n')
    
    
    print(f'New training datasets creation...')
    for donor_datasets in test_datasets_extracted:
        print(f'\nNew Donor')


        gen_results = generate_new_datasets(donor_datasets, n_sub, n_cells, seed)
    
        new_test_datasets += gen_results[0]
        new_test_y += gen_results[1]
        seed += 1 
    print(new_test_y )
    print(f'Done!\n')

    return new_train_datasets, new_train_y, new_val_datasets, new_val_y, new_test_datasets, new_test_y

def remove_labels(new_test_datasets):
    no_labels = []
    for data in new_test_datasets:
        data = dataset.drop(columns = ['IsBlast'])
        no_labels.append(dataset)
    return no_labels

In [31]:

# Samples donors for Train, Validation and Test sets    
train_donors_idx, val_donors_idx, test_donors_idx = dataset_elaboration(multiple_donations, ALL_DATASETS, healthy_donors, blast_donors,
                        mixed_donors, n_sub = 3, seed = 42)



Precess starts. Dividing donors...
healthy_donors_idx, blast_donors_idx, mixed_donors_idx: [0, 4, 2, 1, 3], [0, 2, 1],[4, 0, 2, 1, 3]
Seting Train, Validation and Test idx...
['12', '9', '13', '7', '11'] ['6', '15', '3'] ['2', '8', '14', '1', '4']


In [32]:

#  Retrieves specific donor datasets from all datasets list
train_datasets_extracted = donation_extraction(train_donors_idx, multiple_donations, ALL_DATASETS)
val_datasets_extracted = donation_extraction(val_donors_idx, multiple_donations, ALL_DATASETS)
test_datasets_extracted = donation_extraction(test_donors_idx, multiple_donations, ALL_DATASETS)

#print(len(train_datasets_extracted)) # list of donators
#print(len(train_datasets_extracted[0])) # list of donations
#print(len(train_datasets_extracted[0][0])) # dataset of cells

In [102]:
n_sub = 2
seed = 42
n_cells = 100000
new_train_datasets, new_train_y, new_val_datasets, new_val_y, new_test_datasets, new_test_y = splitting_and_dataset_elaboration(train_datasets_extracted, val_datasets_extracted, test_datasets_extracted, n_sub, n_cells, seed)

new_no_label_test_dataset = remove_labels(new_test_datasets)

New training datasets creation...

New Donor
New Artificial Blast Donation 1: length = 105000, healthy cells:288053, blast cells: 5000
New Artificial Blast Donation 2: length = 101000, healthy cells:288053, blast cells: 1000
New Artificial Healthy Donation 1: length = 100000
New Artificial Healthy Donation 2: length = 100000

New Donor
New Artificial Blast Donation 1: length = 101000, healthy cells:154929, blast cells: 1000
New Artificial Blast Donation 2: length = 105000, healthy cells:154929, blast cells: 5000
New Artificial Healthy Donation 1: length = 100000
New Artificial Healthy Donation 2: length = 100000

New Donor
New Artificial Healthy Donation 1: length = 100000
New Artificial Healthy Donation 2: length = 100000

New Donor
New Artificial Blast Donation 1: length = 100500, healthy cells:131153, blast cells: 500
New Artificial Blast Donation 2: length = 101000, healthy cells:131153, blast cells: 1000
New Artificial Healthy Donation 1: length = 100000
New Artificial Healthy Don

In [12]:
train = new_train_datasets[0]
val = new_val_datasets[0]
test = new_test_datasets[0]

In [103]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
import os

#BASE_DIR = Path("data/raw/")

RESULT_DIR = f'{path_unfixed}/data/B-ALL'

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(f'{RESULT_DIR}/train', exist_ok=True)
os.makedirs(f'{RESULT_DIR}/test', exist_ok=True)

#train_set = pd.read_csv("metadata/B-ALL_train.csv")
#test_set = pd.read_csv("metadata/B-ALL_test.csv")

Index = 0
for i, data in enumerate(new_train_datasets):
    Index += 1
    feature_names = dataset.columns

    b_data = len(dataset[dataset['IsBlast'] == 1])
    proportion = b_data / len(dataset)
    data = dataset.drop(columns = ['IsBlast'])
    
    X = torch.tensor(dataset.to_numpy())
    #X = torch.tensor(dataset)
    y = new_train_y[i]
    data = {
        "X": X,
        "y": y,
        "idx": Index,
        "proportion": proportion,
        "tags": ["B-ALL", "train"],
        "feature_names": feature_names
    }
    path = f'{RESULT_DIR}/train/'+ f"{Index}.pt"

    torch.save(data, path)

for i, data in enumerate(new_test_datasets):
    Index += 1
    #print(i)
    feature_names = dataset.columns

    b_data = len(dataset[dataset['IsBlast'] == 1])
    proportion = b_data / len(dataset)
    data = dataset.drop(columns = ['IsBlast'])
    
    X = torch.tensor(dataset.to_numpy())
    #X = torch.tensor(dataset)
    y = new_train_y[i]
    
    data = {
        "X": X,
        "y": y,
        "idx": Index,
        "proportion": proportion,
        "tags": ["B-ALL", "test"],
        "feature_names": feature_names
    }
    path = f'{RESULT_DIR}/test/'+ f"{Index}.pt"

    torch.save(data, path)

In [97]:
import sys
from yaml import load, FullLoader
sys.argv = ['tuning_ball_logistic.yaml', r'C:\Users\admin\0.Master_Thesis\CSNN\csnn\config\tuning_ball_logistic.yaml']

In [148]:
decache_files = ['trainer', 'cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 'Dataset_Elaboration_Class', 'Dataset_utils', 'cellCnn.downsample_new_unfixed']
# Rimuovi il modulo specifico dalla cache

from Dataset_utils import remove_from_cache
remove_from_cache(decache_files)

from utils import trainer, DensityDifference

importlib.reload(trainer)
importlib.reload(DensityDifference)
importlib.reload(dataset)

from utils import trainer, DensityDifference, dataset
#import train_logistic_cv


from Dataset_Elaboration_Class import class_division, donor_division, splitting, patient_code_extraction
from Dataset_Elaboration_Class import run_CellCNN_res, dataset_elaboration, CellCNN_restructured, CV_CellCNN_restructured, CV_best_acc

from Dataset_utils import extract_hyper, phenotype_prediction, default_serializer, remove_from_cache, show_hyperparameters, min_length
from Dataset_utils import elaborate_predictions, show_hyper

trainer non trovato nella cache
cellCnn.model_new_unfixed non trovato nella cache
cellCnn.utils_new_unfixed non trovato nella cache
Dataset_Elaboration_Class rimosso dalla cache
Dataset_utils rimosso dalla cache
cellCnn.downsample_new_unfixed non trovato nella cache


In [118]:
%%time
import numpy as np
import random
import torch
import csv
import os
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import ParameterGrid
from yaml import load, FullLoader
import sys
from tqdm import tqdm

from utils.dataset import PointsetDataset
from utils.trainer import dd_initialize, train_initialize, train_head, finetune, load_evaluate

with open(sys.argv[1], 'r') as f:
    config = load(f, Loader=FullLoader)
    #print(config)

import os
print(os.path.exists('0.Unfixed_files/data/B-ALL/train'))
print(config['datasets'])

start_seed = config['start_seed']
end_seed = config['start_seed'] + config['n_seeds']

metrics = [
    ("accuracy", accuracy_score),
    ("f1", f1_score),
    ("precision", precision_score),
    ("recall", recall_score),
    ("auc", roc_auc_score),
]


os.makedirs("experiment/%s/" % config['experiment_name'], exist_ok=True)
with open('experiment/%s/results.csv' % config['experiment_name'], 'w') as f:
    writer = csv.writer(f)
    row_names = ["seed"]
    writer.writerow(row_names + list(sample.keys()) + [name + "_train" for name, _ in metrics] + [name + "_valid" for name, _ in metrics])

#print(config['param_grid'])
param_grid = ParameterGrid(config['param_grid']) # makes grid search over all combinations of parameters (originally 80)
sample = param_grid[0]

#param_grid = ParameterGrid(config['param_grid'])
#print(f'tm: {tqdm(param_grid)}')
#print(f'len_tm: {len(tqdm(param_grid))}')

np.random.seed(0)
random.seed(0)
torch.manual_seed(0)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f'start_training...')
for seed in range(start_seed, end_seed):

    # retrieve train and validation data (dataset.py)
    dataset_train = PointsetDataset(config['datasets'], train=True, fold=0, random_state=seed, **config['dataset_config'])
    dataset_valid = PointsetDataset(config['datasets'], train=False, fold=0, random_state=seed, **config['dataset_config'])

    #print(dataset_valid)

    # trains the initial scores using hernel density
    diff, allpos_sample, allneg_sample = dd_initialize(dataset_train, dataset_valid, **config['experiment_config'])
  
    for params in tqdm(param_grid): # for every combination of parameters 
        
        hashparams = params.copy()
        hashparams['architecture'] = frozenset(hashparams['architecture'])
        param_hash = hash(frozenset(hashparams.items()))

        resdir = "experiment/%s/models/seed_%d/hash_%d" % (config['experiment_name'], seed, param_hash) #results directory

        os.makedirs(resdir, exist_ok=True)

        if True:
            kwargs = params.copy()
            for key in config['experiment_config']:
                kwargs[key] = config['experiment_config'][key]

            # train the model
            model, Xinit_train, yinit_train = train_initialize(dataset_train, dataset_valid, diff, allpos_sample, allneg_sample, **kwargs)

            torch.save((Xinit_train, yinit_train), "%s/dd_init.pt" % resdir)
            torch.save(model.cpu(), "%s/model_init.pt" % resdir)

            # train again the model with the best  parameters and weights? 
            model = train_head(dataset_train, dataset_valid, model, **config['experiment_config'])

            torch.save(model.cpu(), "%s/model_head.pt" % resdir)

            
            model, (preds_val, y_valid), (preds_train, y_train) = finetune(dataset_train, dataset_valid, model, **kwargs)

            torch.save(model, "%s/model.pt" % resdir)
        else:
            # validation
            model, (preds_val, y_valid), (preds_train, y_train) = load_evaluate(dataset_train, dataset_valid, "%s/model.pt")
            
        train_metrics = []
        for name, metric in metrics:
            if name in ["accuracy", "f1", "precision", "recall"]:
                train_metrics.append(metric(y_train.numpy(), (preds_train > 0.5).to(int).numpy()))
            else:
                train_metrics.append(metric(y_train.numpy(), preds_train.numpy()))

        val_metrics = []
        for name, metric in metrics:
            if name in ["accuracy", "f1", "precision", "recall"]:
                val_metrics.append(metric(y_valid.numpy(), (preds_val > 0.5).to(int).numpy()))
            else:
                val_metrics.append(metric(y_valid.numpy(), preds_val.numpy()))


        with open('experiment/%s/results.csv' % config['experiment_name'], 'a') as f:
            writer = csv.writer(f)
            metadata = [seed]
            param_list = [str(v) for v in params.values()]
            writer.writerow(metadata + param_list + train_metrics + val_metrics)


False
['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train']


  0%|                                                                                            | 0/8 [00:00<?, ?it/s]

len_tm: 8
start_training...
['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train']
['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\1.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\10.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\11.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\12.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\13.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\14.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\15.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\16.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\17.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/tra

['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train']
['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\1.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\10.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\11.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\12.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\13.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\14.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\15.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\16.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\17.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\18.pt', 'C:\\Users\\admi

100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 49.85it/s]


...

[Epoch 9] tr_loss: 0.6835, tr_acc: 0.5714, te_loss: 0.6979, te_acc: 0.5000
Initializing models with 5 restarts
[Epoch 0] tr_loss: 0.7188, tr_acc: 0.5393
[Epoch 250] tr_loss: 0.1556, tr_acc: 0.9847
[Epoch 500] tr_loss: 0.0712, tr_acc: 0.9840
[Epoch 750] tr_loss: 0.0535, tr_acc: 0.9840
[Epoch 1000] tr_loss: 0.0461, tr_acc: 0.9840
[Epoch 1250] tr_loss: 0.0416, tr_acc: 0.9860
[Epoch 1500] tr_loss: 0.0383, tr_acc: 0.9873
[Epoch 1750] tr_loss: 0.0357, tr_acc: 0.9873
[Epoch 2000] tr_loss: 0.0336, tr_acc: 0.9880
[Epoch 2250] tr_loss: 0.0318, tr_acc: 0.9893
[Epoch 2500] tr_loss: 0.0302, tr_acc: 0.9913
[Epoch 2750] tr_loss: 0.0287, tr_acc: 0.9913
[Epoch 3000] tr_loss: 0.0274, tr_acc: 0.9913
[Epoch 3250] tr_loss: 0.0261, tr_acc: 0.9913
[Epoch 3500] tr_loss: 0.0249, tr_acc: 0.9913
[Epoch 3750] tr_loss: 0.0237, tr_acc: 0.9933
[Epoch 4000] tr_loss: 0.0226, tr_acc: 0.9933
[Epoch 4250] tr_loss: 0.0215, tr_acc: 0.9933
[Epoch 4500] tr_loss: 0.0206, tr_acc: 0.9933
[Epoch 4750] tr_loss: 0.0196, tr_acc: 0.

C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
 88%|████████████████████████████████████████████████████████████████████████▋          | 7/8 [16:51<02:26, 146.30s/it]

[Epoch 34] tr_loss: 0.6836, tr_acc: 0.5714, te_loss: 0.6980, te_acc: 0.5000
Initializing models with 5 restarts
[Epoch 0] tr_loss: 0.6551, tr_acc: 0.6733
[Epoch 250] tr_loss: 0.1973, tr_acc: 0.9800
[Epoch 500] tr_loss: 0.0836, tr_acc: 0.9840
[Epoch 750] tr_loss: 0.0575, tr_acc: 0.9840
[Epoch 1000] tr_loss: 0.0475, tr_acc: 0.9847
[Epoch 1250] tr_loss: 0.0420, tr_acc: 0.9847
[Epoch 1500] tr_loss: 0.0383, tr_acc: 0.9853
[Epoch 1750] tr_loss: 0.0355, tr_acc: 0.9867
[Epoch 2000] tr_loss: 0.0332, tr_acc: 0.9867
[Epoch 2250] tr_loss: 0.0313, tr_acc: 0.9887
[Epoch 2500] tr_loss: 0.0296, tr_acc: 0.9893
[Epoch 2750] tr_loss: 0.0281, tr_acc: 0.9900
[Epoch 3000] tr_loss: 0.0268, tr_acc: 0.9913
[Epoch 3250] tr_loss: 0.0254, tr_acc: 0.9913
[Epoch 3500] tr_loss: 0.0242, tr_acc: 0.9913
[Epoch 3750] tr_loss: 0.0229, tr_acc: 0.9920
[Epoch 4000] tr_loss: 0.0216, tr_acc: 0.9927
[Epoch 4250] tr_loss: 0.0204, tr_acc: 0.9927
[Epoch 4500] tr_loss: 0.0192, tr_acc: 0.9940
[Epoch 4750] tr_loss: 0.0181, tr_acc: 0

C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
100%|███████████████████████████████████████████████████████████████████████████████████| 8/8 [21:02<00:00, 157.82s/it]

CPU times: total: 3h 5min 9s
Wall time: 2h 29min 16s


NameError: name 'dataset' is not defined

In [177]:
decache_files = ['dataset', 'trainer', 'cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 'Dataset_Elaboration_Class', 'Dataset_utils', 'cellCnn.downsample_new_unfixed']
# Rimuovi il modulo specifico dalla cache

from Dataset_utils import remove_from_cache
remove_from_cache(decache_files)

from utils import trainer, DensityDifference, dataset

importlib.reload(trainer)
importlib.reload(DensityDifference)
importlib.reload(dataset)

from utils import trainer, DensityDifference, dataset
#import train_logistic_cv
from utils.dataset import PointsetDataset

from Dataset_Elaboration_Class import class_division, donor_division, splitting, patient_code_extraction
from Dataset_Elaboration_Class import run_CellCNN_res, dataset_elaboration, CellCNN_restructured, CV_CellCNN_restructured, CV_best_acc

from Dataset_utils import extract_hyper, phenotype_prediction, default_serializer, remove_from_cache, show_hyperparameters, min_length
from Dataset_utils import elaborate_predictions, show_hyper

dataset non trovato nella cache
trainer non trovato nella cache
cellCnn.model_new_unfixed non trovato nella cache
cellCnn.utils_new_unfixed non trovato nella cache
Dataset_Elaboration_Class rimosso dalla cache
Dataset_utils rimosso dalla cache
cellCnn.downsample_new_unfixed non trovato nella cache


In [178]:
#dataset_test = PointsetDataset(config['datasets'], train=False, fold=0, random_state=seed) #, **config['dataset_config'])
#print(os.path.exists(f'{path_unfixed}/data/B-ALL/test'))

sys.argv = ['test_ball_logistic.yaml', r'C:\Users\admin\0.Master_Thesis\CSNN\csnn\config\test_ball_logistic.yaml']
with open(sys.argv[1], 'r') as f:
    test_config = load(f, Loader=FullLoader)
    print(test_config)
#test_path = f'{path_unfixed}/data/B-ALL/test'
print(test_config['datasets'])
#FIXARE TESTING
dataset_test = PointsetDataset(test_config['datasets'], train=False, fold=0, random_state=seed) #, **config['dataset_config'])

def test_load_evaluate(dataset_test, fn):
    x_test = dataset_test.X
    y_test = dataset_test.y
    
    x_test = x_test.to(device).float()
    y_test = y_test.to(device).float()

    model = torch.load(fn)
    model = model.to(device)
    model.eval()

    with torch.set_grad_enabled(False):
        preds_test = model(x_test)
        preds_test = preds_test.squeeze(1)

    return model.cpu(), (preds_test.cpu(), y_test.cpu())

model, (preds_test, y_test) = test_load_evaluate(new_test_datasets,  "%s/model.pt")

{'experiment_name': 'tuning_ball_logistic', 'start_seed': 0, 'n_seeds': 1, 'datasets': ['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn/0.Unfixed_files/data/B-ALL/test/']}
['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn/0.Unfixed_files/data/B-ALL/test/']
['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn/0.Unfixed_files/data/B-ALL/test/']
ok
data:C:\Users\admin\0.Master_Thesis\CSNN\csnn/0.Unfixed_files/data/B-ALL/test/
fn: C:\Users\admin\0.Master_Thesis\CSNN\csnn/0.Unfixed_files/data/B-ALL/test\19.pt
fn: C:\Users\admin\0.Master_Thesis\CSNN\csnn/0.Unfixed_files/data/B-ALL/test\20.pt
fn: C:\Users\admin\0.Master_Thesis\CSNN\csnn/0.Unfixed_files/data/B-ALL/test\21.pt
fn: C:\Users\admin\0.Master_Thesis\CSNN\csnn/0.Unfixed_files/data/B-ALL/test\22.pt
fn: C:\Users\admin\0.Master_Thesis\CSNN\csnn/0.Unfixed_files/data/B-ALL/test\23.pt
fn: C:\Users\admin\0.Master_Thesis\CSNN\csnn/0.Unfixed_files/data/B-ALL/test\24.pt
fn: C:\Users\admin\0.Master_Thesis\CSNN\csnn/0.Unfixed_files/data/B-ALL/test\25.pt


UnboundLocalError: local variable 'n_test' referenced before assignment

In [114]:
from tqdm import tqdm
import numpy as np

x_train = np.arange(10)        # finto dataset
y_train = np.array([0,1,0,1,0,1,0,0,1,0])

for x in tqdm(x_train[y_train == 1], disable=False):
    print(f'\n')
    print(x)

100%|██████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 2001.82it/s]



1


3


5


8


In [36]:
alpha = 1
dd_sample_size = 10
# sets parameter for KDE class
dd = DensityDifference.DensityDifference(alpha, sample_size=dd_sample_size)

# Train the KDE
allpos_sample, allneg_sample = dd.fit(train_samples_array, train_y)

1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
ok

ok


  0%|          | 0/20000 [00:00<?, ?it/s]

1
Iteration 0: [0.84600794 0.9671433  0.49374247 2.10809278 0.64026421 1.45579839
 2.82005382 0.69655949 1.24065685 3.12258649 2.71802425]

ok


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
'''
import shutil
import os

def copy_directory_to_drive():
    source_dir = "/content/csnn"
    dest_dir = "/content/drive/MyDrive/Master Thesis/CSNN"

    print(f"Copia di {source_dir} in {dest_dir}...")

    try:
        # Crea la directory di destinazione se non esiste
        os.makedirs(dest_dir, exist_ok=True)

        # Copia i contenuti (se la cartella di destinazione esiste già, shutil.copytree fallisce)
        if os.path.exists(os.path.join(dest_dir, "csnn")):
            print("La cartella csnn esiste già nella destinazione. Eliminando la versione precedente...")
            shutil.rmtree(os.path.join(dest_dir, "csnn"))

        shutil.copytree(source_dir, os.path.join(dest_dir, "csnn"))
        print("Copia completata.")

    except Exception as e:
        print(f"Errore durante la copia: {e}")

# Esegui la funzione
if __name__ == "__main__":
    copy_directory_to_drive()
'''

In [ ]:
# ========================================
# IMPORT CSNN ORIGINALE SU GOOGLE COLAB
# ========================================

# 1. SETUP INIZIALE
print("=== SETUP CSNN SU GOOGLE COLAB ===")

# Clona la repository
!git clone https://github.com/erobl/csnn.git
%cd csnn

# Mostra struttura del progetto
print("\n📁 STRUTTURA PROGETTO:")
!tree -L 2

# 2. INSTALLAZIONE DIPENDENZE
print("\n=== INSTALLAZIONE DIPENDENZE ===")

# Mostra i requirements
print("📋 Requirements originali:")
!cat requirements.txt

# Installa i requirements
!pip install -r requirements.txt

# Verifica versioni installate
print("\n✅ VERIFICA INSTALLAZIONE:")
import torch
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import sklearn
from tqdm import tqdm

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ NumPy: {np.__version__}")
print(f"✓ Pandas: {pd.__version__}")
print(f"✓ Scikit-learn: {sklearn.__version__}")
print(f"✓ CUDA disponibile: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 3. ESPLORAZIONE CODICE
print("\n=== ESPLORAZIONE CODICE ===")

# Mostra file Python principali
print("📝 FILE PYTHON PRINCIPALI:")
!ls -la *.py

# Mostra configurazioni
print("\n⚙️ FILE CONFIGURAZIONE:")
!ls -la config/

# Mostra alcune configurazioni di esempio
print("\n📄 Esempio configurazione B-ALL:")
!head -20 config/best_ball_logistic.yaml

# 4. ANALISI STRUTTURA MODULI
print("\n=== ANALISI MODULI ===")

# Controlla se ci sono moduli custom
import os
import importlib.util

def analyze_python_files():
    python_files = [f for f in os.listdir('.') if f.endswith('.py')]

    for file in python_files:
        print(f"\n📄 {file}:")
        # Mostra prime righe per capire cosa fa
        with open(file, 'r') as f:
            lines = f.readlines()[:10]
            for i, line in enumerate(lines, 1):
                if line.strip().startswith('#') or line.strip().startswith('"""'):
                    print(f"  {i}: {line.rstrip()}")
                elif 'import' in line or 'def ' in line or 'class ' in line:
                    print(f"  {i}: {line.rstrip()}")

analyze_python_files()

# 5. CONTROLLO DIPENDENZE DEL MODELLO
print("\n=== CONTROLLO MODELLO ===")

# Cerca i file che definiscono il modello CSNN
!grep -l "class.*CSNN\|def.*csnn" *.py

# Cerca le funzioni di training
!grep -l "train.*" *.py | head -5

# 6. VERIFICA REQUISITI MINIMI
print("\n=== VERIFICA SISTEMA ===")

import psutil
import sys

# RAM
ram = psutil.virtual_memory()
print(f"💾 RAM: {ram.available/1e9:.1f}/{ram.total/1e9:.1f} GB disponibili")

# CPU
print(f"🖥️ CPU cores: {psutil.cpu_count()}")

# Python
print(f"🐍 Python: {sys.version}")

# Spazio disco
disk = psutil.disk_usage('.')
print(f"💿 Spazio disco: {disk.free/1e9:.1f} GB liberi")

# 7. TEST IMPORT MODULI
print("\n=== TEST IMPORT MODULI ===")

try:
    # Prova ad importare moduli del progetto se esistono
    sys.path.append('.')

    # Lista dei possibili moduli
    possible_modules = ['model', 'utils', 'data_loader', 'train', 'csnn']

    for module_name in possible_modules:
        if os.path.exists(f"{module_name}.py"):
            try:
                module = importlib.import_module(module_name)
                print(f"✓ {module_name}.py importato con successo")

                # Mostra le funzioni/classi principali
                attrs = [attr for attr in dir(module) if not attr.startswith('_')]
                if attrs:
                    print(f"  Contiene: {', '.join(attrs[:5])}")

            except Exception as e:
                print(f"❌ Errore import {module_name}: {str(e)[:50]}...")

except Exception as e:
    print(f"Errore generale: {e}")

# 8. PREPARAZIONE PER STUDIO
print("\n=== PREPARAZIONE STUDIO CODICE ===")

# Crea notebook helper functions
def show_file_content(filename, lines=None):
    """Mostra contenuto di un file"""
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            content = f.readlines()
            if lines:
                content = content[:lines]
            for i, line in enumerate(content, 1):
                print(f"{i:3}: {line.rstrip()}")
    else:
        print(f"File {filename} non trovato")

def find_function(func_name):
    """Trova una funzione nei file Python"""
    python_files = [f for f in os.listdir('.') if f.endswith('.py')]

    for file in python_files:
        with open(file, 'r') as f:
            lines = f.readlines()
            for i, line in enumerate(lines):
                if f"def {func_name}" in line or f"class {func_name}" in line:
                    print(f"📍 Trovato in {file}:{i+1}")
                    print(f"   {line.strip()}")

def list_configs():
    """Lista tutte le configurazioni disponibili"""
    if os.path.exists('config'):
        configs = [f for f in os.listdir('config') if f.endswith('.yaml')]
        for config in configs:
            print(f"⚙️ {config}")

print("🔧 FUNZIONI HELPER DISPONIBILI:")
print("- show_file_content('filename.py', lines=20)")
print("- find_function('function_name')")
print("- list_configs()")

list_configs()

# 9. PRIMO TEST DI FUNZIONAMENTO
print("\n=== PRIMO TEST ===")

# Prova a vedere se riesce a caricare una configurazione
try:
    config_file = 'config/best_ball_logistic.yaml'
    if os.path.exists(config_file):
        with open(config_file, 'r') as f:
            config = yaml.safe_load(f)
        print(f"✓ Configurazione {config_file} caricata:")
        print(f"  Chiavi principali: {list(config.keys())}")
    else:
        print("❌ File configurazione non trovato")

except Exception as e:
    print(f"❌ Errore caricamento config: {e}")

print("\n🎉 SETUP COMPLETATO!")
print("Ora puoi studiare il codice usando:")
print("- show_file_content('train_logistic.py') per vedere il training")
print("- find_function('CSNN') per trovare la definizione del modello")
print("- !python train_logistic.py --help per vedere le opzioni")

In [ ]:
# Vedi tutti i file Python
!ls -la *.py

# Vedi la struttura generale
!head -10 *.py

In [ ]:
# Cerca KDE (Kernel Density Estimation)
!grep -r -A 5 -B 5 "kde\|kernel\|density" *.py

print('ok')
# Cerca il calcolo di Bayes
!grep -r -A 10 "bayes\|probability\|P(" *.py